# Stage 08 — Cross-Method XAI Evaluation

This notebook performs a reproducible, artifact-only evaluation of the completed SHAP, LIME and DiCE stages for the XAI Credit Study. It does not generate explanations, counterfactuals, models or data partitions.

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ARTIFACTS_DIR = Path("../artifacts")
PLOTS_DIR = ARTIFACTS_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
EXPECTED_CASE_IDS = [f"XAI_{number:03d}" for number in range(1, 21)]
ORIGINAL_FEATURES = [
    "NAME_CONTRACT_TYPE", "AMT_INCOME_TOTAL", "AMT_CREDIT",
    "AMT_ANNUITY", "AMT_GOODS_PRICE", "CNT_FAM_MEMBERS",
    "EMPLOYMENT_YEARS", "NAME_INCOME_TYPE", "NAME_HOUSING_TYPE",
    "AMT_REQ_CREDIT_BUREAU_MON", "AMT_REQ_CREDIT_BUREAU_QRT",
    "AMT_REQ_CREDIT_BUREAU_YEAR", "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO", "CREDIT_ANNUITY_RATIO",
]
DISPLAY_NAMES = {
    "NAME_CONTRACT_TYPE": "Contract type",
    "AMT_INCOME_TOTAL": "Annual income",
    "AMT_CREDIT": "Requested credit amount",
    "AMT_ANNUITY": "Annuity amount",
    "AMT_GOODS_PRICE": "Goods price",
    "CNT_FAM_MEMBERS": "Family members",
    "EMPLOYMENT_YEARS": "Employment duration",
    "NAME_INCOME_TYPE": "Income type",
    "NAME_HOUSING_TYPE": "Housing type",
    "AMT_REQ_CREDIT_BUREAU_MON": "Credit enquiries in previous month",
    "AMT_REQ_CREDIT_BUREAU_QRT": "Credit enquiries in previous quarter",
    "AMT_REQ_CREDIT_BUREAU_YEAR": "Credit enquiries in previous year",
    "CREDIT_INCOME_RATIO": "Credit-to-income ratio",
    "ANNUITY_INCOME_RATIO": "Annuity-to-income ratio",
    "CREDIT_ANNUITY_RATIO": "Credit-to-annuity ratio",
}
print("Stage 08 configured for persisted-artifact evaluation only.")

Stage 08 configured for persisted-artifact evaluation only.


## 1. Final Artifact Inventory

The inventory distinguishes required evaluation inputs from unavailable optional diagnostics. No upstream method is rerun when an optional saved diagnostic is absent.

In [ ]:
artifact_spec = {
    "xgboost_credit_model.joblib": "Frozen weighted XGBoost model",
    "preprocessor.joblib": "Frozen preprocessing pipeline",
    "model_metadata.json": "Frozen model configuration and metrics",
    "train_indices.csv": "Saved training partition",
    "test_indices.csv": "Saved test partition",
    "xai_evaluation_cases.csv": "Fixed 20-case XAI cohort",
    "shap_local_explanations.csv": "SHAP local attribution records",
    "shap_top_factors.csv": "SHAP top-five original-feature factors",
    "shap_plain_english.csv": "SHAP narrative explanations",
    "shap_sparsity_metrics.csv": "SHAP compactness metrics",
    "shap_metadata.json": "SHAP provenance",
    "lime_top_factors.csv": "LIME top-five original-feature factors",
    "lime_plain_english.csv": "LIME narrative explanations",
    "lime_sparsity_metrics.csv": "LIME compactness metrics",
    "lime_preliminary_stability.csv": "Saved preliminary LIME stability",
    "lime_preliminary_faithfulness.csv": "Saved preliminary LIME faithfulness",
    "lime_metadata.json": "LIME provenance",
    "dice_counterfactuals.csv": "Validated recovered DiCE counterfactuals",
    "dice_feature_changes.csv": "DiCE actionable-feature audit",
    "dice_sparsity_metrics.csv": "DiCE actionable sparsity",
    "dice_proximity_metrics.csv": "DiCE normalized proximity",
    "dice_plain_english.csv": "DiCE narrative explanations",
    "dice_case_summary.csv": "DiCE case summary",
    "dice_generation_first_pass.csv": "Frozen first-pass outcomes",
    "dice_timeout_retry.csv": "Frozen timeout retry outcomes",
    "dice_final_case_status.csv": "Final 20-case DiCE status",
    "dice_recovery_status.csv": "Counterfactual artifact recovery audit",
    "dice_metadata.json": "DiCE provenance",
}
inventory_rows = []
for artifact, purpose in artifact_spec.items():
    path = ARTIFACTS_DIR / artifact
    rows = np.nan
    if path.exists() and path.suffix == ".csv":
        rows = len(pd.read_csv(path))
    inventory_rows.append({
        "artifact": artifact, "exists": path.exists(), "rows": rows, "purpose": purpose
    })
artifact_inventory = pd.DataFrame(inventory_rows)
display(artifact_inventory)

required = [
    "xai_evaluation_cases.csv", "shap_local_explanations.csv",
    "shap_top_factors.csv",
    "shap_sparsity_metrics.csv", "lime_top_factors.csv",
    "lime_sparsity_metrics.csv", "lime_preliminary_stability.csv",
    "lime_preliminary_faithfulness.csv", "dice_counterfactuals.csv",
    "dice_feature_changes.csv", "dice_sparsity_metrics.csv",
    "dice_proximity_metrics.csv", "dice_final_case_status.csv",
]
missing_required = [name for name in required if not (ARTIFACTS_DIR / name).exists()]
if missing_required:
    raise FileNotFoundError(f"Required finalized artifacts missing: {missing_required}")

SHAP_STABILITY_PATH = ARTIFACTS_DIR / "shap_preliminary_stability.csv"
SHAP_FAITHFULNESS_PATH = ARTIFACTS_DIR / "shap_preliminary_faithfulness.csv"
print(f"Saved SHAP preliminary stability available: {SHAP_STABILITY_PATH.exists()}")
print(f"Saved SHAP preliminary faithfulness available: {SHAP_FAITHFULNESS_PATH.exists()}")

,artifact,exists,rows,purpose
0,xgboost_credit_model.joblib,True,NaN,Frozen weighted XGBoost model
1,preprocessor.joblib,True,NaN,Frozen preprocessing pipeline
2,model_metadata.json,True,NaN,Frozen model configuration and metrics
3,train_indices.csv,True,246008.0,Saved training partition
4,test_indices.csv,True,61503.0,Saved test partition
5,xai_evaluation_cases.csv,True,20.0,Fixed 20-case XAI cohort
6,shap_local_explanations.csv,True,560.0,SHAP local attribution records
7,shap_top_factors.csv,True,100.0,SHAP top-five original-feature factors
8,shap_plain_english.csv,True,20.0,SHAP narrative explanations
9,shap_sparsity_metrics.csv,True,20.0,SHAP compactness metrics


Saved SHAP preliminary stability available: False
Saved SHAP preliminary faithfulness available: False


## 2–3. Common Cases and Canonical Original Features

SHAP and LIME are compared at the original 15-feature level. Transformed dummy columns and raw LIME rule strings are not treated as feature identities.

In [ ]:
cases = pd.read_csv(ARTIFACTS_DIR / "xai_evaluation_cases.csv")
shap_local = pd.read_csv(ARTIFACTS_DIR / "shap_local_explanations.csv")
shap_top = pd.read_csv(ARTIFACTS_DIR / "shap_top_factors.csv")
lime_top = pd.read_csv(ARTIFACTS_DIR / "lime_top_factors.csv")
shap_sparsity = pd.read_csv(ARTIFACTS_DIR / "shap_sparsity_metrics.csv")
lime_sparsity = pd.read_csv(ARTIFACTS_DIR / "lime_sparsity_metrics.csv")
dice_counterfactuals = pd.read_csv(ARTIFACTS_DIR / "dice_counterfactuals.csv")
dice_changes = pd.read_csv(ARTIFACTS_DIR / "dice_feature_changes.csv")
dice_sparsity = pd.read_csv(ARTIFACTS_DIR / "dice_sparsity_metrics.csv")
dice_proximity = pd.read_csv(ARTIFACTS_DIR / "dice_proximity_metrics.csv")
dice_final = pd.read_csv(ARTIFACTS_DIR / "dice_final_case_status.csv")

assert len(cases) == 20 and cases["case_id"].tolist() == EXPECTED_CASE_IDS
assert set(shap_top["case_id"]) == set(EXPECTED_CASE_IDS)
assert set(lime_top["case_id"]) == set(EXPECTED_CASE_IDS)
assert set(dice_final["case_id"]) == set(EXPECTED_CASE_IDS)
assert shap_top.groupby("case_id").size().eq(5).all()
assert lime_top.groupby("case_id").size().eq(5).all()
assert set(shap_top["original_feature"]).issubset(ORIGINAL_FEATURES)
assert set(lime_top["original_feature"]).issubset(ORIGINAL_FEATURES)
assert len(dice_counterfactuals) == 8
assert dice_counterfactuals["valid_counterfactual"].astype(str).str.lower().isin(
    ["true", "1", "1.0"]
).all()
common_cases = cases.copy()
display(common_cases)

,case_id,row_index,true_target,predicted_class,predicted_probability,case_type
0,XAI_001,288644,0,1,0.871334,high-confidence positive
1,XAI_002,234361,0,1,0.861796,high-confidence positive
2,XAI_003,25106,1,1,0.851671,high-confidence positive
3,XAI_004,148440,0,1,0.845883,high-confidence positive
4,XAI_005,267529,0,1,0.844541,high-confidence positive
5,XAI_006,259190,0,1,0.500000,borderline positive
6,XAI_007,125832,0,1,0.500008,borderline positive
7,XAI_008,3508,0,1,0.500023,borderline positive
8,XAI_009,264908,0,1,0.500030,borderline positive
9,XAI_010,253960,0,1,0.500040,borderline positive


## 4. SHAP–LIME Top-Feature Agreement

Agreement is calculated after canonicalization to original features. Spearman rank agreement is reported only when at least two features are shared.

In [ ]:
def ranked_features(frame, case_id, value_column):
    subset = frame.loc[frame["case_id"].eq(case_id)].copy()
    if "rank" in subset:
        subset = subset.sort_values("rank")
    else:
        subset = subset.assign(
            _absolute=subset[value_column].abs()
        ).sort_values("_absolute", ascending=False)
    return subset["original_feature"].tolist()

agreement_rows = []
for case_id in EXPECTED_CASE_IDS:
    shap_features = ranked_features(shap_top, case_id, "shap_value")[:5]
    lime_features = ranked_features(lime_top, case_id, "lime_weight")[:5]
    shap3, lime3 = set(shap_features[:3]), set(lime_features[:3])
    shap5, lime5 = set(shap_features), set(lime_features)
    common = sorted(shap5 & lime5)
    rank_correlation = np.nan
    if len(common) >= 2:
        shap_ranks = pd.Series(
            [shap_features.index(feature) + 1 for feature in common], dtype=float
        ).rank()
        lime_ranks = pd.Series(
            [lime_features.index(feature) + 1 for feature in common], dtype=float
        ).rank()
        if shap_ranks.nunique() > 1 and lime_ranks.nunique() > 1:
            rank_correlation = float(np.corrcoef(shap_ranks, lime_ranks)[0, 1])
    agreement_rows.append({
        "case_id": case_id,
        "top_1_agreement": shap_features[0] == lime_features[0],
        "top_3_overlap_count": len(shap3 & lime3),
        "top_3_jaccard": len(shap3 & lime3) / len(shap3 | lime3),
        "top_5_overlap_count": len(shap5 & lime5),
        "top_5_jaccard": len(shap5 & lime5) / len(shap5 | lime5),
        "shared_top_5_features": "; ".join(common),
        "shared_feature_rank_spearman": rank_correlation,
    })
shap_lime_agreement = pd.DataFrame(agreement_rows)
shap_lime_agreement.to_csv(
    ARTIFACTS_DIR / "xai_shap_lime_agreement.csv", index=False
)
top1_count = int(shap_lime_agreement["top_1_agreement"].sum())
print(f"Mean top-3 overlap: {shap_lime_agreement['top_3_overlap_count'].mean():.3f}")
print(f"Mean top-3 Jaccard: {shap_lime_agreement['top_3_jaccard'].mean():.3f}")
print(f"Mean top-5 overlap: {shap_lime_agreement['top_5_overlap_count'].mean():.3f}")
print(f"Mean top-5 Jaccard: {shap_lime_agreement['top_5_jaccard'].mean():.3f}")
print(f"Top-1 agreements: {top1_count}/20 ({top1_count / 20:.1%})")
display(shap_lime_agreement)

Mean top-3 overlap: 1.800
Mean top-3 Jaccard: 0.475
Mean top-5 overlap: 3.250
Mean top-5 Jaccard: 0.499
Top-1 agreements: 4/20 (20.0%)


,case_id,top_1_agreement,top_3_overlap_count,top_3_jaccard,top_5_overlap_count,top_5_jaccard,shared_top_5_features,shared_feature_rank_spearman
0,XAI_001,False,1,0.2,3,0.428571,AMT_GOODS_PRICE; EMPLOYMENT_YEARS; NAME_INCOME...,0.5
1,XAI_002,False,2,0.5,3,0.428571,AMT_ANNUITY; AMT_GOODS_PRICE; EMPLOYMENT_YEARS,1.0
2,XAI_003,False,2,0.5,2,0.250000,AMT_GOODS_PRICE; EMPLOYMENT_YEARS,-1.0
3,XAI_004,False,2,0.5,3,0.428571,AMT_GOODS_PRICE; EMPLOYMENT_YEARS; NAME_INCOME...,0.5
4,XAI_005,False,0,0.0,3,0.428571,AMT_GOODS_PRICE; EMPLOYMENT_YEARS; NAME_INCOME...,-0.5
5,XAI_006,False,2,0.5,3,0.428571,AMT_GOODS_PRICE; EMPLOYMENT_YEARS; NAME_INCOME...,1.0
6,XAI_007,False,2,0.5,3,0.428571,AMT_GOODS_PRICE; EMPLOYMENT_YEARS; NAME_INCOME...,0.5
7,XAI_008,True,3,1.0,3,0.428571,AMT_GOODS_PRICE; EMPLOYMENT_YEARS; NAME_CONTRA...,0.5
8,XAI_009,True,2,0.5,4,0.666667,AMT_ANNUITY; AMT_CREDIT; AMT_GOODS_PRICE; EMPL...,1.0
9,XAI_010,True,1,0.2,4,0.666667,AMT_ANNUITY; AMT_CREDIT; AMT_GOODS_PRICE; NAME...,0.2


## 5. Directional Agreement

SHAP and LIME have different mathematical semantics. This is therefore a sign-consistency comparison for shared original features, not equality of attribution magnitudes.

In [ ]:
direction_rows = []
direction_comparisons = []
for case_id in EXPECTED_CASE_IDS:
    shap_case = shap_top.loc[shap_top["case_id"].eq(case_id)].set_index("original_feature")
    lime_case = lime_top.loc[lime_top["case_id"].eq(case_id)].set_index("original_feature")
    shared = sorted(set(shap_case.index) & set(lime_case.index))
    same = 0
    for feature in shared:
        shap_positive = float(shap_case.loc[feature, "shap_value"]) > 0
        lime_positive = float(lime_case.loc[feature, "lime_weight"]) > 0
        agrees = shap_positive == lime_positive
        same += int(agrees)
        direction_comparisons.append({
            "case_id": case_id, "feature": feature,
            "shap_positive": shap_positive, "lime_positive": lime_positive,
            "same_direction": agrees,
        })
    direction_rows.append({
        "case_id": case_id,
        "shared_feature_count": len(shared),
        "same_direction_count": same,
        "direction_agreement_rate": same / len(shared) if shared else np.nan,
    })
direction_agreement = pd.DataFrame(direction_rows)
direction_details = pd.DataFrame(direction_comparisons)
direction_agreement.to_csv(
    ARTIFACTS_DIR / "xai_direction_agreement.csv", index=False
)
overall_direction_agreement = (
    direction_details["same_direction"].mean() if len(direction_details) else np.nan
)
print(f"Overall directional agreement: {overall_direction_agreement:.1%}")
display(direction_agreement)

Overall directional agreement: 92.3%


,case_id,shared_feature_count,same_direction_count,direction_agreement_rate
0,XAI_001,3,3,1.000000
1,XAI_002,3,3,1.000000
2,XAI_003,2,2,1.000000
3,XAI_004,3,3,1.000000
4,XAI_005,3,3,1.000000
5,XAI_006,3,3,1.000000
6,XAI_007,3,2,0.666667
7,XAI_008,3,3,1.000000
8,XAI_009,4,4,1.000000
9,XAI_010,4,4,1.000000


## 6. Explanation Compactness

These measures represent different notions of compactness: cumulative attribution mass for SHAP/LIME and actionable changes for DiCE. Fewer features are not assumed to be universally better.

In [ ]:
compactness_sources = [
    ("SHAP", "features_for_50_percent", shap_sparsity["features_for_50_percent"]),
    ("SHAP", "features_for_80_percent", shap_sparsity["features_for_80_percent"]),
    ("LIME", "features_for_50_percent", lime_sparsity["features_for_50_percent"]),
    ("LIME", "features_for_80_percent", lime_sparsity["features_for_80_percent"]),
    (
        "DiCE", "number_of_changed_actionable_features",
        dice_sparsity["number_of_changed_actionable_features"],
    ),
]
sparsity_rows = []
for method, metric, values in compactness_sources:
    values = pd.to_numeric(values, errors="coerce").dropna()
    sparsity_rows.append({
        "method": method, "metric": metric, "count": len(values),
        "mean": values.mean(), "median": values.median(),
        "min": values.min(), "max": values.max(),
    })
sparsity_comparison = pd.DataFrame(sparsity_rows)
sparsity_comparison.to_csv(
    ARTIFACTS_DIR / "xai_sparsity_comparison.csv", index=False
)
display(sparsity_comparison)

,method,metric,count,mean,median,min,max
0,SHAP,features_for_50_percent,20,3.050,3.0,2,4
1,SHAP,features_for_80_percent,20,6.700,7.0,4,9
2,LIME,features_for_50_percent,20,1.900,2.0,1,2
3,LIME,features_for_80_percent,20,3.100,3.0,3,4
4,DiCE,number_of_changed_actionable_features,8,1.875,1.5,1,3


## 7–8. Preliminary Stability and Faithfulness

Saved diagnostics are summarized without rerunning perturbations. SHAP preliminary stability and faithfulness are unavailable because those artifacts were never persisted. LIME surrogate fidelity is reported separately from intervention sensitivity; surrogate R² is not equated with SHAP faithfulness.

In [ ]:
lime_stability = pd.read_csv(
    ARTIFACTS_DIR / "lime_preliminary_stability.csv"
)
stability_rows = [{
    "method": "SHAP", "cases_tested": 0,
    "mean_top_5_overlap": np.nan, "median_top_5_overlap": np.nan,
    "minimum_top_5_overlap": np.nan, "maximum_top_5_overlap": np.nan,
    "availability": "unavailable: no persisted SHAP stability artifact",
}]
lime_overlap = pd.to_numeric(lime_stability["top_5_overlap_count"])
stability_rows.append({
    "method": "LIME", "cases_tested": len(lime_overlap),
    "mean_top_5_overlap": lime_overlap.mean(),
    "median_top_5_overlap": lime_overlap.median(),
    "minimum_top_5_overlap": lime_overlap.min(),
    "maximum_top_5_overlap": lime_overlap.max(),
    "availability": "saved preliminary diagnostic",
})
stability_comparison = pd.DataFrame(stability_rows)
stability_comparison.to_csv(
    ARTIFACTS_DIR / "xai_stability_comparison.csv", index=False
)

lime_faithfulness = pd.read_csv(
    ARTIFACTS_DIR / "lime_preliminary_faithfulness.csv"
)
faithfulness_rows = [{
    "method": "SHAP", "cases_tested": 0,
    "mean_absolute_score_change": np.nan,
    "median_absolute_score_change": np.nan,
    "min_absolute_score_change": np.nan,
    "max_absolute_score_change": np.nan,
    "availability": "unavailable: no persisted SHAP faithfulness artifact",
}]
lime_changes = pd.to_numeric(lime_faithfulness["absolute_score_change"])
faithfulness_rows.append({
    "method": "LIME", "cases_tested": len(lime_changes),
    "mean_absolute_score_change": lime_changes.mean(),
    "median_absolute_score_change": lime_changes.median(),
    "min_absolute_score_change": lime_changes.min(),
    "max_absolute_score_change": lime_changes.max(),
    "availability": "saved preliminary diagnostic",
})
faithfulness_comparison = pd.DataFrame(faithfulness_rows)
faithfulness_comparison.to_csv(
    ARTIFACTS_DIR / "xai_faithfulness_comparison.csv", index=False
)

lime_fidelity_cases = (
    lime_top[[
        "case_id", "predicted_probability", "lime_local_prediction",
        "local_fidelity_r2",
    ]].drop_duplicates("case_id").copy()
)
lime_fidelity_cases["absolute_local_prediction_error"] = (
    lime_fidelity_cases["predicted_probability"]
    - lime_fidelity_cases["lime_local_prediction"]
).abs()
fidelity_rows = []
for metric in ["local_fidelity_r2", "absolute_local_prediction_error"]:
    values = pd.to_numeric(lime_fidelity_cases[metric])
    fidelity_rows.append({
        "metric": metric, "count": len(values), "mean": values.mean(),
        "median": values.median(), "min": values.min(), "max": values.max(),
    })
lime_local_fidelity_summary = pd.DataFrame(fidelity_rows)
lime_local_fidelity_summary.to_csv(
    ARTIFACTS_DIR / "xai_lime_local_fidelity_summary.csv", index=False
)
display(stability_comparison)
display(faithfulness_comparison)
display(lime_local_fidelity_summary)

,method,cases_tested,mean_top_5_overlap,median_top_5_overlap,minimum_top_5_overlap,maximum_top_5_overlap,availability
0,SHAP,0,NaN,NaN,NaN,NaN,unavailable: no persisted SHAP stability artifact
1,LIME,5,5.0,5.0,5.0,5.0,saved preliminary diagnostic


,method,cases_tested,mean_absolute_score_change,median_absolute_score_change,min_absolute_score_change,max_absolute_score_change,availability
0,SHAP,0,NaN,NaN,NaN,NaN,unavailable: no persisted SHAP faithfulness ar...
1,LIME,5,0.176504,0.188229,0.138632,0.197027,saved preliminary diagnostic


,metric,count,mean,median,min,max
0,local_fidelity_r2,20,0.324156,0.248255,0.129397,0.552847
1,absolute_local_prediction_error,20,0.130461,0.108408,0.002543,0.283717


## 9. DiCE Counterfactual Evaluation

A completed search with no returned counterfactual means: “No valid counterfactual was returned within the configured DiCE search and actionability constraints.” It does not establish mathematical non-existence. A timeout means only that the allocated computational budget was exhausted.

In [ ]:
dice_eval = (
    dice_counterfactuals.merge(
        dice_sparsity[[
            "case_id", "counterfactual_id",
            "number_of_changed_actionable_features",
        ]],
        on=["case_id", "counterfactual_id"], how="left",
    ).merge(
        dice_proximity, on=["case_id", "counterfactual_id"], how="left"
    )
)
dice_eval["absolute_probability_change"] = (
    dice_eval["counterfactual_probability"]
    - dice_eval["original_probability"]
).abs()
dice_eval.to_csv(ARTIFACTS_DIR / "xai_dice_evaluation.csv", index=False)

dice_status_counts = dice_final["final_status"].value_counts()
assert dice_status_counts.get("success_first_pass", 0) == 8
assert dice_status_counts.get("no_counterfactual_returned_after_retry", 0) == 9
assert dice_status_counts.get("timeout_after_retry", 0) == 1
assert dice_status_counts.get("technical_error_first_pass", 0) == 2
class_1_to_0 = int(
    ((dice_eval["original_class"] == 1) & (dice_eval["counterfactual_class"] == 0)).sum()
)
class_0_to_1 = int(
    ((dice_eval["original_class"] == 0) & (dice_eval["counterfactual_class"] == 1)).sum()
)
print("Evaluation cases: 20")
print("Valid counterfactual cases: 8")
print("No-counterfactual-returned-after-retry: 9")
print("Timeout-after-retry: 1")
print("Technical-error-first-pass: 2")
print("Counterfactual availability: 8/20 (40.0%)")
print(f"Successful class 1 -> class 0 cases: {class_1_to_0}")
print(f"Successful class 0 -> class 1 cases: {class_0_to_1}")
display(dice_eval)

Evaluation cases: 20
Valid counterfactual cases: 8
No-counterfactual-returned-after-retry: 9
Timeout-after-retry: 1
Technical-error-first-pass: 2
Counterfactual availability: 8/20 (40.0%)
Successful class 1 -> class 0 cases: 7
Successful class 0 -> class 1 cases: 1


,case_id,counterfactual_id,original_probability,original_class,counterfactual_probability,counterfactual_class,desired_class,valid_counterfactual,number_of_changed_features,automatically_recomputed_derived_features,proximity_score_x,permitted_ranges_respected,derived_ratios_consistent,actionability_pass,plausibility_pass,failure_reason,number_of_changed_actionable_features,proximity_score_y,absolute_probability_change
0,XAI_001,XAI_001_RECOVERED_CF01,0.871334,1,0.229084,0,0,True,3,CREDIT_INCOME_RATIO; ANNUITY_INCOME_RATIO; CRE...,3.451275,True,True,True,True,NaN,3,3.451275,0.642250
1,XAI_002,XAI_002_RECOVERED_CF01,0.861796,1,0.389667,0,0,True,2,CREDIT_INCOME_RATIO; ANNUITY_INCOME_RATIO; CRE...,1.396619,True,True,True,True,NaN,2,1.396619,0.472128
2,XAI_003,XAI_003_RECOVERED_CF01,0.851671,1,0.286159,0,0,True,1,CREDIT_INCOME_RATIO; ANNUITY_INCOME_RATIO; CRE...,1.386415,True,True,True,True,NaN,1,1.386415,0.565512
3,XAI_004,XAI_004_RECOVERED_CF01,0.845883,1,0.398225,0,0,True,1,CREDIT_INCOME_RATIO; ANNUITY_INCOME_RATIO; CRE...,1.123663,True,True,True,True,NaN,1,1.123663,0.447658
4,XAI_005,XAI_005_RECOVERED_CF01,0.844541,1,0.418978,0,0,True,1,CREDIT_INCOME_RATIO; ANNUITY_INCOME_RATIO; CRE...,1.130878,True,True,True,True,NaN,1,1.130878,0.425563
5,XAI_006,XAI_006_RECOVERED_CF01,0.500000,1,0.426312,0,0,True,3,CREDIT_INCOME_RATIO; ANNUITY_INCOME_RATIO; CRE...,0.932460,True,True,True,True,NaN,3,0.932460,0.073688
6,XAI_009,XAI_009_RECOVERED_CF01,0.500030,1,0.325985,0,0,True,3,CREDIT_INCOME_RATIO; ANNUITY_INCOME_RATIO; CRE...,4.678333,True,True,True,True,NaN,3,4.678333,0.174045
7,XAI_017,XAI_017_RECOVERED_CF01,0.499978,0,0.607299,1,1,True,1,CREDIT_INCOME_RATIO; ANNUITY_INCOME_RATIO; CRE...,0.275442,True,True,True,True,NaN,1,0.275442,0.107321


## 10–11. Cross-Method Case Summary and Case-Type Comparison

Case-type comparisons are descriptive because the 20 cases were intentionally selected and constitute a small, non-random evaluation cohort. No significance testing is performed.

In [ ]:
shap_first = shap_top.sort_values(["case_id", "rank"]).groupby("case_id").first()
lime_first = lime_top.sort_values(["case_id", "rank"]).groupby("case_id").first()
cross_method = cases.copy()
cross_method["shap_top_feature"] = cross_method["case_id"].map(
    shap_first["display_feature"]
)
cross_method["shap_top_direction"] = cross_method["case_id"].map(
    shap_first["direction"]
)
cross_method["lime_top_feature"] = cross_method["case_id"].map(
    lime_first["display_feature"]
)
cross_method["lime_top_direction"] = cross_method["case_id"].map(
    lime_first["direction"]
)
cross_method = cross_method.merge(
    shap_lime_agreement[[
        "case_id", "top_1_agreement", "top_5_overlap_count", "top_5_jaccard"
    ]], on="case_id", how="left"
).merge(
    direction_agreement[["case_id", "direction_agreement_rate"]],
    on="case_id", how="left"
).merge(
    dice_final[[
        "case_id", "valid_counterfactual_available", "final_status"
    ]].rename(columns={
        "valid_counterfactual_available": "dice_valid_counterfactual",
        "final_status": "dice_final_status",
    }), on="case_id", how="left"
).merge(
    dice_eval[[
        "case_id", "counterfactual_probability",
        "number_of_changed_actionable_features",
    ]].rename(columns={
        "counterfactual_probability": "dice_counterfactual_probability",
        "number_of_changed_actionable_features":
            "dice_changed_actionable_features",
    }), on="case_id", how="left"
)
assert len(cross_method) == 20 and cross_method["case_id"].is_unique
cross_method.to_csv(
    ARTIFACTS_DIR / "xai_cross_method_case_summary.csv", index=False
)

case_metrics = (
    cross_method.merge(
        shap_sparsity.add_prefix("shap_"),
        left_on="case_id", right_on="shap_case_id", how="left",
    ).merge(
        lime_sparsity.add_prefix("lime_"),
        left_on="case_id", right_on="lime_case_id", how="left",
    )
)
case_type_rows = []
for case_type, group in case_metrics.groupby("case_type", sort=False):
    valid_dice = group["dice_valid_counterfactual"].astype(str).str.lower().isin(
        ["true", "1", "1.0"]
    )
    case_type_rows.append({
        "case_type": case_type,
        "case_count": len(group),
        "mean_top_5_overlap_count": group["top_5_overlap_count"].mean(),
        "mean_top_5_jaccard": group["top_5_jaccard"].mean(),
        "mean_direction_agreement_rate": group["direction_agreement_rate"].mean(),
        "mean_shap_features_for_50_percent":
            group["shap_features_for_50_percent"].mean(),
        "mean_shap_features_for_80_percent":
            group["shap_features_for_80_percent"].mean(),
        "mean_lime_features_for_50_percent":
            group["lime_features_for_50_percent"].mean(),
        "mean_lime_features_for_80_percent":
            group["lime_features_for_80_percent"].mean(),
        "dice_counterfactual_availability_rate": valid_dice.mean(),
        "mean_dice_changed_actionable_features": (
            group.loc[valid_dice, "dice_changed_actionable_features"].mean()
        ),
    })
case_type_comparison = pd.DataFrame(case_type_rows)
case_type_comparison.to_csv(
    ARTIFACTS_DIR / "xai_case_type_comparison.csv", index=False
)
display(cross_method)
display(case_type_comparison)

,case_id,row_index,true_target,predicted_class,predicted_probability,case_type,shap_top_feature,shap_top_direction,lime_top_feature,lime_top_direction,top_1_agreement,top_5_overlap_count,top_5_jaccard,direction_agreement_rate,dice_valid_counterfactual,dice_final_status,dice_counterfactual_probability,dice_changed_actionable_features
0,XAI_001,288644,0,1,0.871334,high-confidence positive,Credit-to-annuity ratio,increases model risk output,Contract type,increases local higher-risk prediction,False,3,0.428571,1.000000,True,success_first_pass,0.229084,3.0
1,XAI_002,234361,0,1,0.861796,high-confidence positive,Credit-to-annuity ratio,increases model risk output,Contract type,increases local higher-risk prediction,False,3,0.428571,1.000000,True,success_first_pass,0.389667,2.0
2,XAI_003,25106,1,1,0.851671,high-confidence positive,Credit-to-annuity ratio,increases model risk output,Contract type,increases local higher-risk prediction,False,2,0.250000,1.000000,True,success_first_pass,0.286159,1.0
3,XAI_004,148440,0,1,0.845883,high-confidence positive,Credit-to-annuity ratio,increases model risk output,Contract type,increases local higher-risk prediction,False,3,0.428571,1.000000,True,success_first_pass,0.398225,1.0
4,XAI_005,267529,0,1,0.844541,high-confidence positive,Credit-to-annuity ratio,increases model risk output,Contract type,increases local higher-risk prediction,False,3,0.428571,1.000000,True,success_first_pass,0.418978,1.0
5,XAI_006,259190,0,1,0.500000,borderline positive,Employment duration,decreases model risk output,Contract type,increases local higher-risk prediction,False,3,0.428571,1.000000,True,success_first_pass,0.426312,3.0
6,XAI_007,125832,0,1,0.500008,borderline positive,Credit-to-annuity ratio,increases model risk output,Contract type,increases local higher-risk prediction,False,3,0.428571,0.666667,False,technical_error_first_pass,NaN,NaN
7,XAI_008,3508,0,1,0.500023,borderline positive,Contract type: Cash loans,decreases model risk output,Contract type,decreases local higher-risk prediction,True,3,0.428571,1.000000,False,no_counterfactual_returned_after_retry,NaN,NaN
8,XAI_009,264908,0,1,0.500030,borderline positive,Goods price,decreases model risk output,Goods price,decreases local higher-risk prediction,True,4,0.666667,1.000000,True,success_first_pass,0.325985,3.0
9,XAI_010,253960,0,1,0.500040,borderline positive,Goods price,decreases model risk output,Goods price,decreases local higher-risk prediction,True,4,0.666667,1.000000,False,no_counterfactual_returned_after_retry,NaN,NaN


,case_type,case_count,mean_top_5_overlap_count,mean_top_5_jaccard,mean_direction_agreement_rate,mean_shap_features_for_50_percent,mean_shap_features_for_80_percent,mean_lime_features_for_50_percent,mean_lime_features_for_80_percent,dice_counterfactual_availability_rate,mean_dice_changed_actionable_features
0,high-confidence positive,5,2.8,0.392857,1.000000,3.0,6.8,2.0,3.0,1.0,1.6
1,borderline positive,5,3.4,0.523810,0.933333,3.4,7.2,1.8,3.2,0.4,3.0
2,high-confidence negative,5,3.6,0.571429,0.783333,2.6,5.8,2.0,3.0,0.0,NaN
3,borderline negative,5,3.2,0.507143,1.000000,3.2,7.0,1.8,3.2,0.2,1.0


## 12. Dissertation-Quality Visualizations

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

plt.figure(figsize=(11, 5))
plt.bar(shap_lime_agreement["case_id"], shap_lime_agreement["top_5_jaccard"], color="#315E8A")
plt.ylim(0, 1)
plt.ylabel("Top-5 Jaccard")
plt.xlabel("Evaluation case")
plt.title("SHAP–LIME Top-5 Original-Feature Agreement")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "xai_shap_lime_top5_agreement.png", dpi=180)
plt.close()

plt.figure(figsize=(11, 5))
plt.bar(direction_agreement["case_id"], direction_agreement["direction_agreement_rate"], color="#4D9078")
plt.ylim(0, 1)
plt.ylabel("Direction agreement rate")
plt.xlabel("Evaluation case")
plt.title("SHAP–LIME Directional Consistency for Shared Top-5 Features")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "xai_direction_agreement.png", dpi=180)
plt.close()

plot_sparsity = sparsity_comparison.copy()
plot_sparsity["label"] = (
    plot_sparsity["method"] + " — "
    + plot_sparsity["metric"].str.replace("_", " ")
)
plt.figure(figsize=(10, 5))
plt.barh(plot_sparsity["label"], plot_sparsity["mean"], color=["#315E8A", "#5B8DB8", "#C17C45", "#D99A63", "#4D9078"])
plt.xlabel("Mean feature count")
plt.title("Method-Specific Explanation Compactness")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "xai_sparsity_comparison.png", dpi=180)
plt.close()

faith_plot = faithfulness_comparison.dropna(
    subset=["mean_absolute_score_change"]
)
plt.figure(figsize=(7, 4.5))
plt.bar(faith_plot["method"], faith_plot["mean_absolute_score_change"], color="#C17C45")
plt.ylabel("Mean absolute probability change")
plt.title("Saved Preliminary Intervention Sensitivity")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "xai_faithfulness_comparison.png", dpi=180)
plt.close()

outcome_labels = [
    "Valid CF", "No CF returned\nafter retry",
    "Timeout\nafter retry", "Technical error\nfirst pass",
]
outcome_counts = [8, 9, 1, 2]
plt.figure(figsize=(9, 5))
plt.bar(outcome_labels, outcome_counts, color=["#4D9078", "#C9A227", "#C17C45", "#A64B4B"])
plt.ylabel("Cases")
plt.title("Final DiCE Outcomes Across 20 Evaluation Cases")
for index, value in enumerate(outcome_counts):
    plt.text(index, value + 0.15, str(value), ha="center")
plt.ylim(0, max(outcome_counts) + 1.5)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "xai_dice_outcomes.png", dpi=180)
plt.close()
print("Created five Stage 08 plots.")

Created five Stage 08 plots.


## Interpretation of Cross-Method XAI Evaluation

**SHAP.** SHAP attributes model output to features using an additive attribution framework and supports both global and local interpretation. The saved compactness results describe how concentrated absolute attribution mass is across features. No persisted preliminary SHAP stability or faithfulness diagnostic exists, so those measures are reported as unavailable rather than recomputed.

**LIME.** LIME approximates the model locally with a surrogate. Agreement with SHAP is evaluated using canonical original-feature rankings and attribution signs. Disagreement is not automatically an error: the methods answer related but mathematically different questions. Local fidelity R² and local prediction error describe the surrogate approximation and are kept separate from intervention sensitivity.

**DiCE.** DiCE provides an actionable counterfactual perspective under deliberately conservative constraints. Successful counterfactuals are evaluated by availability, probability change, proximity and the number of changed actionable features. A completed search with no returned result, or a timeout, must not be interpreted as proof that a feasible counterfactual does not exist.

These descriptive results concern model explanations for an intentionally selected evaluation cohort. They do not establish fairness, causality, regulatory compliance, robustness in deployment or real-world creditworthiness.

## 14–17. Metrics Summary, Metadata and Final Quality Checks

In [ ]:
summary_rows = []
def add_metric(category, metric, method, value, cases_evaluated, interpretation):
    summary_rows.append({
        "category": category, "metric": metric, "method": method,
        "value": value, "cases_evaluated": cases_evaluated,
        "interpretation": interpretation,
    })

add_metric("agreement", "top_1_agreement_rate", "SHAP-LIME", top1_count / 20, 20, "Same highest-ranked original feature")
add_metric("agreement", "mean_top_5_overlap", "SHAP-LIME", shap_lime_agreement["top_5_overlap_count"].mean(), 20, "Shared original features among top five")
add_metric("agreement", "mean_top_5_jaccard", "SHAP-LIME", shap_lime_agreement["top_5_jaccard"].mean(), 20, "Set overlap normalized by union")
add_metric("direction", "overall_direction_agreement", "SHAP-LIME", overall_direction_agreement, len(direction_details), "Sign consistency for shared top-five features")
for row in sparsity_comparison.itertuples(index=False):
    add_metric("sparsity", f"mean_{row.metric}", row.method, row.mean, row.count, "Method-specific compactness; not a universal quality ranking")
for row in stability_comparison.itertuples(index=False):
    add_metric("stability", "mean_top_5_overlap", row.method, row.mean_top_5_overlap, row.cases_tested, row.availability)
for row in faithfulness_comparison.itertuples(index=False):
    add_metric("faithfulness", "mean_absolute_score_change", row.method, row.mean_absolute_score_change, row.cases_tested, row.availability)
for row in lime_local_fidelity_summary.itertuples(index=False):
    add_metric("local_fidelity", f"mean_{row.metric}", "LIME", row.mean, row.count, "Local surrogate approximation diagnostic")
add_metric("counterfactual", "availability_rate", "DiCE", 8 / 20, 20, "Valid returned counterfactuals under configured constraints")

xai_evaluation_summary = pd.DataFrame(summary_rows)
xai_evaluation_summary.to_csv(
    ARTIFACTS_DIR / "xai_evaluation_summary.csv", index=False
)

created_metrics = [
    "xai_shap_lime_agreement.csv", "xai_direction_agreement.csv",
    "xai_sparsity_comparison.csv", "xai_stability_comparison.csv",
    "xai_faithfulness_comparison.csv",
    "xai_lime_local_fidelity_summary.csv", "xai_dice_evaluation.csv",
    "xai_cross_method_case_summary.csv", "xai_case_type_comparison.csv",
    "xai_evaluation_summary.csv",
]
plot_filenames = [
    "plots/xai_shap_lime_top5_agreement.png",
    "plots/xai_direction_agreement.png",
    "plots/xai_sparsity_comparison.png",
    "plots/xai_faithfulness_comparison.png",
    "plots/xai_dice_outcomes.png",
]
metadata = {
    "evaluation_case_count": 20,
    "shap_case_count": int(shap_top["case_id"].nunique()),
    "lime_case_count": int(lime_top["case_id"].nunique()),
    "dice_case_count": int(dice_final["case_id"].nunique()),
    "valid_dice_counterfactual_count": len(dice_counterfactuals),
    "artifact_filenames_used": sorted(artifact_spec),
    "unavailable_saved_metrics": [
        "SHAP preliminary stability",
        "SHAP preliminary faithfulness",
    ],
    "metrics_generated": created_metrics,
    "plot_filenames": plot_filenames,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "random_generation_performed": False,
    "model_retrained": False,
    "preprocessor_refitted": False,
}
with (ARTIFACTS_DIR / "xai_evaluation_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

assert len(cases) == 20
assert shap_top["case_id"].nunique() == 20
assert lime_top["case_id"].nunique() == 20
assert dice_final["case_id"].nunique() == 20
assert len(dice_counterfactuals) == 8
assert len(dice_changes) == 24
assert len(cross_method) == 20 and cross_method["case_id"].is_unique
assert np.allclose(
    shap_local.groupby("case_id")["predicted_probability"].first().loc[EXPECTED_CASE_IDS],
    cases.set_index("case_id").loc[EXPECTED_CASE_IDS, "predicted_probability"],
    atol=1e-7,
)
assert np.allclose(
    lime_top.groupby("case_id")["predicted_probability"].first().loc[EXPECTED_CASE_IDS],
    cases.set_index("case_id").loc[EXPECTED_CASE_IDS, "predicted_probability"],
    atol=1e-7,
)
notebook_text = Path("08_xai_evaluation.ipynb").read_text(encoding="utf-8")
for prohibited in [
    "Tree" + "Explainer(", "shap_" + "values(", "LimeTabular" + "Explainer(",
    "explain_" + "instance(", "generate_" + "counterfactuals(",
    "model." + "fit(", "preprocessor." + "fit(",
    "fit_" + "transform(", "train_" + "test_split(",
]:
    assert prohibited not in notebook_text

print(f"XAI cases evaluated: {len(cases)}")
print(f"SHAP cases: {shap_top['case_id'].nunique()}")
print(f"LIME cases: {lime_top['case_id'].nunique()}")
print(f"DiCE status cases: {dice_final['case_id'].nunique()}")
print(f"Valid DiCE counterfactuals: {len(dice_counterfactuals)}")
print(f"SHAP-LIME top-1 agreement: {top1_count}/20 ({top1_count / 20:.1%})")
print(f"Mean SHAP-LIME top-5 overlap: {shap_lime_agreement['top_5_overlap_count'].mean():.3f}")
print(f"Mean SHAP-LIME top-5 Jaccard: {shap_lime_agreement['top_5_jaccard'].mean():.3f}")
print(f"Overall directional agreement: {overall_direction_agreement:.1%}")
print("SHAP preliminary stability: unavailable (not persisted)")
print(f"LIME preliminary stability mean overlap: {lime_overlap.mean():.3f}/5")
print("SHAP preliminary faithfulness mean score change: unavailable (not persisted)")
print(f"LIME preliminary faithfulness mean score change: {lime_changes.mean():.6f}")
print(f"LIME mean local fidelity R2: {lime_fidelity_cases['local_fidelity_r2'].mean():.6f}")
print(f"LIME mean absolute local prediction error: {lime_fidelity_cases['absolute_local_prediction_error'].mean():.6f}")
print("DiCE counterfactual availability: 8/20 (40.0%)")
print(f"Mean changed actionable features for valid DiCE CFs: {dice_sparsity['number_of_changed_actionable_features'].mean():.3f}")
print("Model retrained: False")
print("Preprocessor refitted: False")
print("SHAP regenerated: False")
print("LIME regenerated: False")
print("DiCE regenerated: False")

XAI cases evaluated: 20
SHAP cases: 20
LIME cases: 20
DiCE status cases: 20
Valid DiCE counterfactuals: 8
SHAP-LIME top-1 agreement: 4/20 (20.0%)
Mean SHAP-LIME top-5 overlap: 3.250
Mean SHAP-LIME top-5 Jaccard: 0.499
Overall directional agreement: 92.3%
SHAP preliminary stability: unavailable (not persisted)
LIME preliminary stability mean overlap: 5.000/5
SHAP preliminary faithfulness mean score change: unavailable (not persisted)
LIME preliminary faithfulness mean score change: 0.176504
LIME mean local fidelity R2: 0.324156
LIME mean absolute local prediction error: 0.130461
DiCE counterfactual availability: 8/20 (40.0%)
Mean changed actionable features for valid DiCE CFs: 1.875
Model retrained: False
Preprocessor refitted: False
SHAP regenerated: False
LIME regenerated: False
DiCE regenerated: False
